# Carbon stock prediction

This script was developed to predict carbon stock in the upper 30 and 100 cm of the sediment of a seagrass bed based on the dataset produced in the script "Environmental_covariates_matching_VRE.R", called "SG_modeling_dataframe.csv"

In [1]:
## Open needed packages
library(here)
library(tidyverse)
library(RNetCDF) 
library(mgcv)
#library(ggplot2)
#library(gridExtra)
#library(rnaturalearth)
#library(rnaturalearthdata)
#library(tidyr)
#library(terra)
#library(viridis)
#library(sf)
library(dplyr)
library(stringr)

# Set working directory
setwd(here::here())

here() starts at /home/jovyan/T5.3_BlueCarbon

── Attaching core tidyverse packages ─────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.0     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ───────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: nlme


Attaching package: ‘nlme’


The following object is masked from ‘package:dplyr’:

    collapse


This is mgcv 1.9-4. For overview type '?mgcv'.



# Input

* `file_SG_modeling_dataframe`, SG_modeling_dataframe.csv, prepared in `T5.3_step01-Env_cov_matching.ipynb`.
* `file_For_modeling_df_shallow_carbon_density`, For_modeling_df_shallow_carbon_density.rds, from [EURO-CARBON](https://zenodo.org/records/14905489)
* `file_GAM_top_reduced_SGstock`, GAM_top_reduced_model_SGstock.rds, carbon density prediction model (prepared in script: `SG_carbonstock_model.R`)

In [2]:
# Open SG_modeling_dataframe.csv
file_SG_modeling_dataframe = "data/Debug-SG_modeling_dataframe.csv"

# Open csv
SG_modeling_dataframe <- read.csv(file_SG_modeling_dataframe)

# check that structure is correct (all should be numeric, except for seagrass_species which should be a factor)
str(SG_modeling_dataframe)

'data.frame':	30 obs. of  13 variables:
 $ latitude                         : num  56.1 56.1 56.1 56.1 56.1 ...
 $ longitude                        : num  14.7 14.7 14.7 14.7 14.7 ...
 $ seagrass_species                 : chr  "Unspecified" "Unspecified" "Unspecified" "Unspecified" ...
 $ bottomT_p95_C_closest            : num  18.1 18.1 18.1 18.1 18.1 ...
 $ uo_mean_1.5m_m_s_closest         : num  -0.00426 -0.00426 -0.00426 -0.00426 -0.00426 ...
 $ vo_p90_1.5m_m_s_closest          : num  0.0902 0.0902 0.0902 0.0902 0.0902 ...
 $ po4_mean_1.5m_mmol_m3_closest    : num  0.142 0.142 0.142 0.142 0.142 ...
 $ pH_mean_1.5m_closest             : num  8.08 8.08 8.08 8.08 8.08 ...
 $ wave_height_VHM0_p95_m_closest   : num  1.37 1.37 1.37 1.37 1.37 ...
 $ Surf_fgco2_p95_molC_m2_yr_closest: num  4.67 4.67 4.67 4.67 4.67 ...
 $ KD_closest                       : num  1.89 1.89 1.89 1.89 1.89 ...
 $ RRS443_closest                   : num  0.000438 0.000438 0.000438 0.000438 0.000438 ...
 $ sedimen

In [3]:
# Prepare dataframe
SG_modeling_dataframe <- SG_modeling_dataframe %>%
  mutate(seagrass_species = str_trim(seagrass_species),  # remove leading/trailing spaces
         seagrass_species = str_squish(seagrass_species))  # remove extra internal spaces

# SG_modeling_dataframe$seagrass_species <- as.factor(SG_modeling_dataframe$seagrass_species)
SG_modeling_dataframe$sediment_mean_depth_cm <- as.numeric(SG_modeling_dataframe$sediment_mean_depth_cm)
str(SG_modeling_dataframe)

'data.frame':	30 obs. of  13 variables:
 $ latitude                         : num  56.1 56.1 56.1 56.1 56.1 ...
 $ longitude                        : num  14.7 14.7 14.7 14.7 14.7 ...
 $ seagrass_species                 : chr  "Unspecified" "Unspecified" "Unspecified" "Unspecified" ...
 $ bottomT_p95_C_closest            : num  18.1 18.1 18.1 18.1 18.1 ...
 $ uo_mean_1.5m_m_s_closest         : num  -0.00426 -0.00426 -0.00426 -0.00426 -0.00426 ...
 $ vo_p90_1.5m_m_s_closest          : num  0.0902 0.0902 0.0902 0.0902 0.0902 ...
 $ po4_mean_1.5m_mmol_m3_closest    : num  0.142 0.142 0.142 0.142 0.142 ...
 $ pH_mean_1.5m_closest             : num  8.08 8.08 8.08 8.08 8.08 ...
 $ wave_height_VHM0_p95_m_closest   : num  1.37 1.37 1.37 1.37 1.37 ...
 $ Surf_fgco2_p95_molC_m2_yr_closest: num  4.67 4.67 4.67 4.67 4.67 ...
 $ KD_closest                       : num  1.89 1.89 1.89 1.89 1.89 ...
 $ RRS443_closest                   : num  0.000438 0.000438 0.000438 0.000438 0.000438 ...
 $ sedimen

In [4]:
# Load
file_For_modeling_df_shallow_carbon_density = "data/For_modeling_df_shallow_carbon_density.rds"

For_modeling_df_shallow_carbon_density <- read_rds(file_For_modeling_df_shallow_carbon_density)
str(For_modeling_df_shallow_carbon_density$random_core_variable)  # 382 levels (from 461 cores in full dataset, down to 382 unique cores)

 Factor w/ 382 levels "Ærøya_Ærøya_NA_2017_8_7_58.417_8.763_3",..: 33 33 33 43 43 43 45 45 45 39 ...


In [5]:
# Check sepcies
"Zostera marina and Zostera noltei" %in% levels(For_modeling_df_shallow_carbon_density$seagrass_species)
setdiff(unique(SG_modeling_dataframe$seagrass_species), levels(For_modeling_df_shallow_carbon_density$seagrass_species))

# From SG_modeling_dataframe
raw_SG <- charToRaw(as.character(SG_modeling_dataframe$seagrass_species[SG_modeling_dataframe$seagrass_species == "Zostera marina/Zostera noltei"][1]))

# From For_modeling_df_shallow_carbon_density
raw_training <- charToRaw(as.character(For_modeling_df_shallow_carbon_density$seagrass_species[For_modeling_df_shallow_carbon_density$seagrass_species == "Zostera marina/Zostera noltei"][1]))
identical(raw_SG, raw_training)

[1] TRUE

character(0)

[1] TRUE

In [6]:
# Match factor levels for seagrass_species
SG_modeling_dataframe$seagrass_species <- factor(
  SG_modeling_dataframe$seagrass_species,
  levels = levels(For_modeling_df_shallow_carbon_density$seagrass_species)
)

str(SG_modeling_dataframe)

'data.frame':	30 obs. of  13 variables:
 $ latitude                         : num  56.1 56.1 56.1 56.1 56.1 ...
 $ longitude                        : num  14.7 14.7 14.7 14.7 14.7 ...
 $ seagrass_species                 : Factor w/ 8 levels "Cymodocea nodosa",..: 4 4 4 4 4 4 4 4 4 4 ...
 $ bottomT_p95_C_closest            : num  18.1 18.1 18.1 18.1 18.1 ...
 $ uo_mean_1.5m_m_s_closest         : num  -0.00426 -0.00426 -0.00426 -0.00426 -0.00426 ...
 $ vo_p90_1.5m_m_s_closest          : num  0.0902 0.0902 0.0902 0.0902 0.0902 ...
 $ po4_mean_1.5m_mmol_m3_closest    : num  0.142 0.142 0.142 0.142 0.142 ...
 $ pH_mean_1.5m_closest             : num  8.08 8.08 8.08 8.08 8.08 ...
 $ wave_height_VHM0_p95_m_closest   : num  1.37 1.37 1.37 1.37 1.37 ...
 $ Surf_fgco2_p95_molC_m2_yr_closest: num  4.67 4.67 4.67 4.67 4.67 ...
 $ KD_closest                       : num  1.89 1.89 1.89 1.89 1.89 ...
 $ RRS443_closest                   : num  0.000438 0.000438 0.000438 0.000438 0.000438 ...
 $ sedime

In [7]:
# Use an existing placeholder level from the training data for random_core_variable
# Doesn't matter which one you use because this will be excluded in the model prediction, so I have just picked one
file_GAM_top_reduced_SGstock = "GAM_top_reduced_model_SGstock.rds"

SG_modeling_dataframe$random_core_variable <- 
  factor(rep("Baltic Sea_Furumon_NA_2021_NA_NA_56.0953_14.7202_1.5", nrow(SG_modeling_dataframe)), 
         levels = levels(For_modeling_df_shallow_carbon_density$random_core_variable))
summary(SG_modeling_dataframe)

# Open carbon density prediction model (prepared in script: SG_carbonstock_model.R)
GAM_top_reduced_SGstock <- readRDS(file_GAM_top_reduced_SGstock)
summary(GAM_top_reduced_SGstock)

# Predict carbon density for new samples based on input variables in the "SG_modeling_dataframe.csv" spreadsheet
predicted_carbon_density <- predict(GAM_top_reduced_SGstock,
                                    newdata = SG_modeling_dataframe, 
                                    type = "response",
                                    exclude = "s(random_core_variable)")
# predicted_carbon_density_coreID_notexcluded <- predict(GAM_top_reduced_SGstock, newdata = SG_modeling_dataframe, type = "response")
# predicted_carbon_density_nocoreID <- predict(GAM_top_reduced_SGstock_nocoreID, newdata = SG_modeling_dataframe, type = "response")

# Add predictions to dataframe
SG_modeling_dataframe$predicted_carbon_density <- predicted_carbon_density
summary(SG_modeling_dataframe)

    latitude       longitude                                seagrass_species
 Min.   :40.62   Min.   : 0.6587   Cymodocea nodosa                 :10     
 1st Qu.:40.62   1st Qu.: 0.6587   Unspecified                      :10     
 Median :42.43   Median :14.7202   Zostera marina and Zostera noltei:10     
 Mean   :46.38   Mean   :14.3435   Halophila stipulacea             : 0     
 3rd Qu.:56.10   3rd Qu.:27.6516   Posidonia oceanica               : 0     
 Max.   :56.10   Max.   :27.6516   Zostera marina                   : 0     
                                   (Other)                          : 0     
 bottomT_p95_C_closest uo_mean_1.5m_m_s_closest vo_p90_1.5m_m_s_closest
 Min.   :18.07         Min.   :-4.260e-03       Min.   :0.00354        
 1st Qu.:18.07         1st Qu.:-4.260e-03       1st Qu.:0.00354        
 Median :23.67         Median : 5.516e-05       Median :0.02197        
 Mean   :23.01         Mean   : 1.562e-03       Mean   :0.03858        
 3rd Qu.:27.29         3


Family: Gamma 
Link function: log 

Formula:
carbon_density_g_c_cm3 ~ s(KD_closest) + s(RRS443_closest) + 
    s(wave_height_VHM0_p95_m_closest) + s(po4_mean_1.5m_mmol_m3_closest) + 
    s(pH_mean_1.5m_closest) + s(bottomT_p95_C_closest) + s(vo_p90_1.5m_m_s_closest) + 
    s(uo_mean_1.5m_m_s_closest) + s(Surf_fgco2_p95_molC_m2_yr_closest) + 
    s(sediment_mean_depth_cm) + seagrass_species + s(random_core_variable, 
    bs = "re")

Parametric coefficients:
                                                    Estimate Std. Error t value
(Intercept)                                         -4.62666    0.15861 -29.169
seagrass_speciesHalophila stipulacea                 0.31750    0.31210   1.017
seagrass_speciesPosidonia oceanica                   0.92490    0.16380   5.646
seagrass_speciesUnspecified                         -0.05392    0.17672  -0.305
seagrass_speciesZostera marina                      -0.39057    0.22145  -1.764
seagrass_speciesZostera marina and Cymodocea nodosa -0.688

    latitude       longitude                                seagrass_species
 Min.   :40.62   Min.   : 0.6587   Cymodocea nodosa                 :10     
 1st Qu.:40.62   1st Qu.: 0.6587   Unspecified                      :10     
 Median :42.43   Median :14.7202   Zostera marina and Zostera noltei:10     
 Mean   :46.38   Mean   :14.3435   Halophila stipulacea             : 0     
 3rd Qu.:56.10   3rd Qu.:27.6516   Posidonia oceanica               : 0     
 Max.   :56.10   Max.   :27.6516   Zostera marina                   : 0     
                                   (Other)                          : 0     
 bottomT_p95_C_closest uo_mean_1.5m_m_s_closest vo_p90_1.5m_m_s_closest
 Min.   :18.07         Min.   :-4.260e-03       Min.   :0.00354        
 1st Qu.:18.07         1st Qu.:-4.260e-03       1st Qu.:0.00354        
 Median :23.67         Median : 5.516e-05       Median :0.02197        
 Mean   :23.01         Mean   : 1.562e-03       Mean   :0.03858        
 3rd Qu.:27.29         3

# calculate carbon stock

Carbon stock = Carbon density (gC/cm3) x Depth (cm) x 10,000

Note: 
* 1 hectare = 10,000 square metres (m²), Equivalent to 2.471 acres
* Carbon stocks are often expressed as: tonnes of carbon per hectare (t C/ha) OR megagrams of carbon per hectare (Mg C/ha) (1 Mg = 1 tonne)
* Carbon density is in: gC cm-3

In [8]:
# Calculate carbon stock in the upper 30 and 100 cm

# Calculate carbon stock per 10 cm
# Carbon stock (MgC/ha)=Carbon density (gC/cm³) × Depth (cm) × 100,000,000 (cm2/ha) ÷ 1,000,000 (g/Mg)
# Where:
#   1 hectare = 10,000 m² = 100,000,000 cm²
#   1 Mg = 1,000,000 g
SG_modeling_dataframe$predicted_carbon_stock_per_10cm = 
  SG_modeling_dataframe$predicted_carbon_density*10*100000000*(1/1000000)

# Create sample_ID column
SG_modeling_dataframe$sample_ID <- rep(1:(nrow(SG_modeling_dataframe)/10), each = 10)

# Calculate carbon stock in upper 30 and 100 cm 
SG_modeling_dataframe <- SG_modeling_dataframe %>%
  group_by(sample_ID) %>%
  mutate(
    carbon_stock_Mg_ha_upper30cm  = sum(predicted_carbon_stock_per_10cm[1:3]),
    carbon_stock_Mg_ha_upper100cm = sum(predicted_carbon_stock_per_10cm)
  ) %>%
  ungroup()

str(SG_modeling_dataframe)

tibble [30 × 19] (S3: tbl_df/tbl/data.frame)
 $ latitude                         : num [1:30] 56.1 56.1 56.1 56.1 56.1 ...
 $ longitude                        : num [1:30] 14.7 14.7 14.7 14.7 14.7 ...
 $ seagrass_species                 : Factor w/ 8 levels "Cymodocea nodosa",..: 4 4 4 4 4 4 4 4 4 4 ...
 $ bottomT_p95_C_closest            : num [1:30] 18.1 18.1 18.1 18.1 18.1 ...
 $ uo_mean_1.5m_m_s_closest         : num [1:30] -0.00426 -0.00426 -0.00426 -0.00426 -0.00426 ...
 $ vo_p90_1.5m_m_s_closest          : num [1:30] 0.0902 0.0902 0.0902 0.0902 0.0902 ...
 $ po4_mean_1.5m_mmol_m3_closest    : num [1:30] 0.142 0.142 0.142 0.142 0.142 ...
 $ pH_mean_1.5m_closest             : num [1:30] 8.08 8.08 8.08 8.08 8.08 ...
 $ wave_height_VHM0_p95_m_closest   : num [1:30] 1.37 1.37 1.37 1.37 1.37 ...
 $ Surf_fgco2_p95_molC_m2_yr_closest: num [1:30] 4.67 4.67 4.67 4.67 4.67 ...
 $ KD_closest                       : num [1:30] 1.89 1.89 1.89 1.89 1.89 ...
 $ RRS443_closest                   

# Output

The summary

In [9]:
# Create summary sentences
summary_sentences <- SG_modeling_dataframe %>%
  group_by(sample_ID) %>%
  slice(1) %>%  # take one row per sample
  mutate(summary = paste0(
    "For the seagrass bed at latitude ", latitude, " and longitude ", longitude, "\n",
    "The species identity ", seagrass_species, "\n",
    "The predicted carbon stock in the upper \n",
    "  - upper 30cm  of the sediment is ", format(carbon_stock_Mg_ha_upper30cm, digits=3, nsmall=3),  " Mg/ha\n",
    "  - upper 100cm of the sediment is ", format(carbon_stock_Mg_ha_upper100cm, digits=3, nsmall=3), " Mg/ha\n"
  )) %>%
  pull(summary)

# Print the summaries
cat(summary_sentences, sep = "\n")

For the seagrass bed at latitude 56.0953 and longitude 14.7202
The species identity Unspecified
The predicted carbon stock in the upper 
  - upper 30cm  of the sediment is 13.998 Mg/ha
  - upper 100cm of the sediment is 49.741 Mg/ha

For the seagrass bed at latitude 40.6248 and longitude 0.658723
The species identity Cymodocea nodosa
The predicted carbon stock in the upper 
  - upper 30cm  of the sediment is 13.650 Mg/ha
  - upper 100cm of the sediment is 48.504 Mg/ha

For the seagrass bed at latitude 42.42777 and longitude 27.6516
The species identity Zostera marina and Zostera noltei
The predicted carbon stock in the upper 
  - upper 30cm  of the sediment is 3.748 Mg/ha
  - upper 100cm of the sediment is 13.318 Mg/ha

